In [25]:
import pandas as pd
import json
import ollama
import os
import chromadb
# from chromadb.utils.embedding_functions import HuggingFaceEmbeddingFunction
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer

from sql_metadata import Parser

In [ ]:
# 1. Load the exported CSVs
df_schema = pd.read_csv('adventure_works_full.csv',names = ['schema_name', 'table_name','table_description','column_name','data_type','max_length','column_description'])
df_rel = pd.read_csv('adventure_works_relationships.csv',names = ['table_name', 'column_name','referenced_table_name', 'referenced_column_name'])

# Pre-process relationships into a lookup dictionary
rel_dict = {}
for _, row in df_rel.iterrows():
    tbl = row['table_name']
    ref_tbl = row['referenced_table_name']
    col = row['column_name']
    ref_col = row['referenced_column_name']
    
    # Store relationship from both directions (outgoing and incoming)
    if tbl not in rel_dict: rel_dict[tbl] = []
    if ref_tbl not in rel_dict: rel_dict[ref_tbl] = []
    
    rel_dict[tbl].append(f"{col} -> {ref_tbl}.{ref_col}")
    rel_dict[ref_tbl].append(f"Referenced by {tbl}.{col} via {ref_col}")

schema_summary = {}
grouped = df_schema.groupby(['schema_name', 'table_name'])

for (schema, table_name), group in grouped:
    full_table_name = f"{schema}.{table_name}"
    
    # Get descriptions
    raw_desc = group['table_description'].dropna().unique()
    table_desc = str(raw_desc[0]) if len(raw_desc) > 0 else "None"
    
    # Collect columns
    columns_list = []
    for _, row in group.iterrows():
        columns_list.append(f"{row['column_name']} ({row['data_type']}) - ({row['column_description']})")
    columns_str = ", ".join(columns_list)
    
    # Generate fallback description if missing
    if table_desc in ["None", "nan", ""]:
        prompt = f"Based on the table name '{full_table_name}' and columns [{columns_str}], write a one-sentence summary of what this table tracks."
        response = ollama.chat(model='granite4.1:8b', messages=[{"role": "user", "content": prompt}])
        table_desc = response['message']['content'].strip()

    # Collect matching relationships
    relationships = rel_dict.get(full_table_name, ["No explicit foreign key relationships"])
    
    # 2. Store as structured objects (Best for downstream RAG and matching)
    schema_summary[full_table_name] = {
        "description": table_desc,
        "columns": columns_list,
        "relationships": relationships
    }

schema_summary


{'HumanResources.Department': {'description': 'Lookup table containing the departments within the Adventure Works Cycles company.',
  'columns': ['DepartmentID (smallint) - (Primary key for Department records.)',
   'Name (Name) - (Name of the department.)',
   'GroupName (Name) - (Name of the group to which the department belongs.)',
   'ModifiedDate (datetime) - (Date and time the record was last updated.)'],
  'relationships': ['Referenced by HumanResources.EmployeeDepartmentHistory.DepartmentID via DepartmentID']},
 'HumanResources.Employee': {'description': 'Employee information such as salary, department, and title.',
  'columns': ['BusinessEntityID (int) - (Primary key for Employee records.  Foreign key to BusinessEntity.BusinessEntityID.)',
   'NationalIDNumber (nvarchar) - (Unique national identification number such as a social security number.)',
   'LoginID (nvarchar) - (Network login.)',
   'OrganizationNode (hierarchyid) - (Where the employee is located in corporate hierar

In [ ]:
# # # 3. Save the clean context file
# with open('schema/schema_summary_new.json', 'w') as f:
#     json.dump(schema_summary, f, indent=4)

# print("Successfully compiled schema with relationship mappings!")

Successfully compiled schema with relationship mappings!


In [13]:
with open('schema/schema_summary_new.json','r') as f:
    full_schema_summary = json.load(f)

# full_schema_summary

In [ ]:
# 1. Create a directory to store the documents if it doesn't already exist
output_dir = "schema_documents"
os.makedirs(output_dir, exist_ok=True)

for key, val in full_schema_summary.items():
    schema_catalog = ''

    schema_catalog += f'Table: {key}\n'
    schema_catalog += f'Description: {val["description"]}\n'
    
    # Cleaning up the column and relationship lists for better readability in the text files
    columns_str = ", ".join(val['columns'])
    relationships_str = "\n".join(val['relationships'])
    
    schema_catalog += f'Columns: {columns_str}\n'
    schema_catalog += f'Relationships:\n{relationships_str}\n'

    # 2. Make the table name safe for filenames (replacing the schema dot '.' with an underscore '_')
    # Example: "Sales.Customer" becomes "Sales_Customer.txt"
    safe_filename = f"{key.replace('.', '_')}.txt"
    file_path = os.path.join(output_dir, safe_filename)

    # 3. Write the content to the file
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(schema_catalog)

print(f"Successfully generated {len(full_schema_summary)} schema documents inside the '{output_dir}/' directory!")

Successfully generated 71 schema documents inside the 'schema_documents/' directory!


#### Convert the docs into embeddings and store in Chroma DB

In [7]:
# save vectors permanently
chroma_client = chromadb.PersistentClient(path = "adventure_works_vector_db")

model_name = 'BAAI/bge-large-en-v1.5'
embedding_function = SentenceTransformerEmbeddingFunction(model_name=model_name)

collection = chroma_client.get_or_create_collection(name = "adventure_works_schema",
                                                    embedding_function = embedding_function,
                                                    metadata={"hnsw:space":"cosine"})

# read and insert files into chroma db
docs_dir = "schema_documents"

documents = []
metadatas = []
ids = []

print('Reading schema files...')
for filename in os.listdir(docs_dir):
    if filename.endswith(".txt"):
        file_path = os.path.join(docs_dir, filename)

        # get true file name
        table_name = filename.replace(".txt", "").replace("_",".")

        with open(file_path , 'r', encoding="utf-8") as f:
            file_content = f.read()

        documents.append(file_content)
        metadatas.append({"table_name": table_name, "source":filename})
        ids.append(table_name) # Using table name as the unique ID makes lookups fast

if documents:
    print(f'Embedding and adding {len(documents)} tables to ChromaDB...')
    collection.add(documents = documents,
                   metadatas = metadatas,
                   ids = ids)
    print("Ingestion Complete! Vector DB is ready.")
else:
    print("No documents found to process.")

e:\SQL QUERY GENERATOR\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nishc\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5624.36it/s]


Reading schema files...
Embedding and adding 71 tables to ChromaDB...
Ingestion Complete! Vector DB is ready.


In [ ]:
# # retrive the top-N tables based on user's query

# def retrieve_relevant_tables(user_question, top_n = 4):

#     # connect to persistent database folder
#     client = chromadb.PersistentClient(path = "adventure_works_vector_db")

#     collection = client.get_collection(name="adventure_works_schema")

#     results = collection.query(
#         query_texts = [user_question],
#         n_results = top_n
#     )

#     print(results)
#     # Extract the matched text content blocks and table names
#     matched_schemas = results['documents'][0]
#     matched_tables = [meta['table_name'] for meta in results['metadatas'][0]]

#     # combine the matched schemas intoa tightly pruned string context block
#     pruned_schema_context = "\n\n".join(matched_schemas)

#     return pruned_schema_context, matched_tables

# # context , tables = retrieve_relevant_tables("Show me total revenue for each customer store")

{'ids': [['Sales.Customer', 'Sales.Store', 'Sales.SalesPerson', 'Production.ProductCostHistory']], 'embeddings': None, 'documents': [['Table: Sales.Customer\nDescription: Current customer information. Also see the Person and Store tables.\nColumns: CustomerID (int) - (Primary key.), PersonID (int) - (Foreign key to Person.BusinessEntityID), StoreID (int) - (Foreign key to Store.BusinessEntityID), TerritoryID (int) - (ID of the territory in which the customer is located. Foreign key to SalesTerritory.SalesTerritoryID.), AccountNumber (varchar) - (Unique number identifying the customer assigned by the accounting system.), rowguid (uniqueidentifier) - (ROWGUIDCOL number uniquely identifying the record. Used to support a merge replication sample.), ModifiedDate (datetime) - (Date and time the record was last updated.)\nRelationships:\nTerritoryID -> Sales.SalesTerritory.TerritoryID\nStoreID -> Sales.Store.BusinessEntityID\nPersonID -> Person.Person.BusinessEntityID\nReferenced by Sales.Sal

In [ ]:
# print('Selected tables for the prompt:', tables)

Selected tables for the prompt: ['Sales.Customer', 'Sales.Store', 'Sales.SalesPerson', 'Production.ProductCostHistory']


In [ ]:
# context

'Table: Sales.Customer\nDescription: Current customer information. Also see the Person and Store tables.\nColumns: CustomerID (int) - (Primary key.), PersonID (int) - (Foreign key to Person.BusinessEntityID), StoreID (int) - (Foreign key to Store.BusinessEntityID), TerritoryID (int) - (ID of the territory in which the customer is located. Foreign key to SalesTerritory.SalesTerritoryID.), AccountNumber (varchar) - (Unique number identifying the customer assigned by the accounting system.), rowguid (uniqueidentifier) - (ROWGUIDCOL number uniquely identifying the record. Used to support a merge replication sample.), ModifiedDate (datetime) - (Date and time the record was last updated.)\nRelationships:\nTerritoryID -> Sales.SalesTerritory.TerritoryID\nStoreID -> Sales.Store.BusinessEntityID\nPersonID -> Person.Person.BusinessEntityID\nReferenced by Sales.SalesOrderHeader.CustomerID via CustomerID\n\n\nTable: Sales.Store\nDescription: Customers (resellers) of Adventure Works products.\nColu

In [22]:
# def get_compact_schema(list_of_tables):
#     '''Take a list of tables and return the associated compact summary(description, column names) of each table.'''
#     result = ''

#     try:
#         # Adjusted path to match your JSON location
#         with open('schema_new/schema_summary_new.json', 'r') as f:
#             full_compact_schema_dict = json.load(f)
#     except Exception as e:
#         return f'No schema summary available. Error: {e}'
    
#     for table in list_of_tables:
#         if table not in full_compact_schema_dict:
#             result += f"Table: {table} (Not found in schema catalog)\n\n"
#             continue
            
#         # 1. Clear and collect columns specifically for THIS table
#         col_list = []
#         table_columns = full_compact_schema_dict[table]['columns']
#         for col in table_columns:
#             # Splits at ' - ' to remove the description suffix, then strip spaces
#             clean_col = col.split('-')[0].strip()
#             col_list.append(clean_col)  
            
#         # 2. Join the column list with commas
#         final_col_list = ', '.join(col_list)
        
#         # 3. Fetch the table description
#         table_description = full_compact_schema_dict[table]['description']

#         # 4. Construct a structured layout for this table
#         result += f"Table: {table}\n"
#         result += f"Description: {table_description}\n"
#         result += f"Columns: {final_col_list}\n"
#         result += "-" * 40 + "\n" # Add a divider between tables

#     return result

# get_compact_schema(["HumanResources.Employee", "HumanResources.Department"])

In [28]:
sql = "SELECT e.BusinessEntityID, e.JobTitle FROM HumanResources.Employee e JOIN HumanResources.EmployeeDepartmentHistory edh ON e.BusinessEntityID = edh.BusinessEntityID JOIN HumanResources.Department d ON edh.DepartmentID = d.DepartmentID WHERE d.Name = 'HR' AND e.SickLeaveHours > 10;"

parser = Parser(sql)
parser.columns_dict

{'select': ['HumanResources.Employee.BusinessEntityID',
  'HumanResources.Employee.JobTitle'],
 'join': ['HumanResources.Employee.BusinessEntityID',
  'HumanResources.EmployeeDepartmentHistory.BusinessEntityID',
  'HumanResources.EmployeeDepartmentHistory.DepartmentID',
  'HumanResources.Department.DepartmentID'],
 'where': ['HumanResources.Department.Name',
  'HumanResources.Employee.SickLeaveHours']}